# Milvus的基本使用

# 1、DDL操作

## 1.1 数据库相关操作

### ① 查看数据库

举例1：操作客户端

In [ ]:
from networkx.algorithms.reciprocity import reciprocity
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

举例2：列出所有数据库

In [ ]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

### ② 创建数据库

In [ ]:
db_name = "rag_demo"

if db_name not in existed_databases:
    client.create_database(db_name = db_name)

### ③ 删除数据库

如果数据库下有Collection则无法删除，需要先删除它的所有Collection才能删除Database

In [ ]:
client.drop_database(db_name = db_name)

## 1.2 Collection相关操作

### ① 切换数据库

In [ ]:
client.use_database(db_name = db_name)

### ② 查看数据库下的collections

In [ ]:
collections = client.list_collections()

for coll in collections:
    print(coll)

### ③ 创建collection

In [ ]:
collection_name = "docs"

client.create_collection(
    collection_name = collection_name,
    dimension = 1024,
    metric_type = "COSINE"
)

### ④ 删除collection

In [ ]:
client.drop_collection(
    collection_name = collection_name
)

# 2、DML操作

## 2.1 嵌入模型的初始化

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True)

embedding_model = DashScopeEmbeddings(
	model="qwen3.7-text-embedding",
	dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
)

## 2.2 准备collection

### ① 创建collection

In [ ]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

### ② 查看collection元数据

In [ ]:
from rich import print as rprint

metadata = client.describe_collection(collection_name = collection_name)

rprint(metadata)

## 2.3 准备数据

### ① 准备原始数据

In [ ]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

### ② 生成嵌入向量

In [ ]:
vectors = embedding_model.embed_documents(texts)

### ③ 查看生成的嵌入向量

In [ ]:
print(len(vectors))
print(len(vectors[0]))
print(vectors[0][:5])

### ④ 封装为可以插入的数据格式

In [ ]:
data = [
    {
        "id": i,
        "vector": vectors[i],
        "text": texts[i],
        "source": "demo"
    } for i in range(len(texts))
]

## 2.4 写入数据

### ① 插入数据

In [ ]:
insert_res = client.upsert(
    collection_name = collection_name,
    data = data,

)

print("insert result:", insert_res)

### ② 手动flush

Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

In [ ]:
client.flush(collection_name = collection_name)

### ③ 查看collection统计信息

In [ ]:
status = client.get_collection_stats(
    collection_name = collection_name
)

print("status:", status)

# 3、DQL操作

## 3.1 扫描数据

In [ ]:
iterator = client.query_iterator(
    collection_name = collection_name,
    filter = "",
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)
        print(f"id: {row["id"]}, vector: {row["vector"][:5]}, text: {row["text"]}")

        i += 1

iterator.close()

## 3.2 通过主键查询数据

In [ ]:

res = client.get(
    collection_name = collection_name,
    ids = [0, 1, 2]
)
print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id: {res["id"]}, vector: {res["vector"][:5]}, text: {res["text"]}")

    # print(res[i])

## 3.3 相似度检索

### ① 准备查询嵌入

In [46]:
# 相似度检索
query = "什么是向量数据库"
query_vector = embedding_model.embed_query(query)

### ② 检索

In [48]:
results = client.search(
    collection_name = collection_name,
    limit = 3,
    data = [query_vector],
    output_fields = ["text", "source", "id"]
)

for res in results[0]:
    print(res)

{'id': 1, 'distance': 0.7061716914176941, 'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}}
{'id': 2, 'distance': 0.373348206281662, 'entity': {'id': 2, 'text': 'RAG 的核心是先检索相关知识，再让大模型生成答案。', 'source': 'demo'}}
{'id': 3, 'distance': 0.3724326491355896, 'entity': {'id': 3, 'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
